# Create STAC for dataproduct "vern"

Load YAML config and run feature_class_layers.

In [41]:
from pathlib import Path
from src.yaml_config import load_map_service_config, feature_class_layers

from datetime import datetime, timezone
import pystac

In [42]:
config_path = Path("./config/vern_arcgis_map_service_definition.yml").resolve()
config, config_map, config_layers = load_map_service_config(config_path)

print(f"Loaded: {config_path}")
print(f"Map name: {config_map.get('name')}")
print(f"Total layer entries: {len(config_layers)}")
print(f"Feature class layers: {len(fc_layers)}")

Loaded: C:\Users\WILACA\git\kartai\skygeo\src\stac_structure\config\vern_arcgis_map_service_definition.yml
Map name: vern
Total layer entries: 8
Feature class layers: 6


In [43]:
fc_layers = feature_class_layers(config_layers)
print(f"Feature class layers: {len(fc_layers)}")

for layer_name, layer_cfg in fc_layers:
    service_id = layer_cfg.get("service_id")
    print(f"- service_id={service_id}, layer={layer_name}")

Feature class layers: 6
- service_id=0, layer=naturvern_omraade
- service_id=1, layer=naturvern_klasser_omraade
- service_id=2, layer=naturvern_teiggrensepunkt
- service_id=3, layer=naturvern_grense
- service_id=4, layer=foreslaatt_naturvern_omraade
- service_id=5, layer=foreslaatt_naturvern_grense


In [44]:
# Inspect the first feature class layer config
first_name, first_cfg = fc_layers[0]
print(first_name)
first_cfg

naturvern_omraade


{'type': 'feature_class',
 'source': {'feature_dataset': 'sde_feature_dataset',
  'dataset': 'sde_dataset'},
 'service_id': 0,
 'visible': True,
 'display_field': 'naturvern_id',
 'cols': ['naturvern_id',
  'cdda_id',
  'navn',
  'offisielt_navn',
  'faktaark',
  'verneform',
  'verneforskrift',
  'vernedato',
  'foerstegang_vernet',
  'verneplan',
  'kommune',
  'forvaltningsmyndighet',
  'forvaltningsmyndighet_type',
  'iucn',
  'revisjon',
  'truet_vurdering',
  'major_ecosystem_type',
  'verneform_aggregert',
  'objekttype',
  'objectid',
  'globalid',
  'shape',
  'st_area(shape)',
  'st_perimeter(shape)']}

## Create STAC item

Get collection metadata from YAML

In [45]:
# Fixed requirements from your decision
item_id = "vern_norge_20260615"
generation_time = datetime.now(timezone.utc)

collection_id = "vern"
description = config_map.get("metadata", {}).get("description", "Vernedata")
license_name = config_map.get("metadata", {}).get("license", "proprietary")

service_name = str(config_map.get("name", "vern")).strip()
base_url = "https://vsepublicstorage.blob.core.windows.net/vse-public/stac/miljodir-stac/collections"
lyrx_asset_base = f"{base_url}/vern/assets/lyrx"

bbox = [27250.06038219, 6579917.07024015, 1167250.06038219, 7939917.07024015]
geom = {
    "type": "Polygon",
    "coordinates": [[
        [bbox[0], bbox[1]],
        [bbox[0], bbox[3]],
        [bbox[2], bbox[3]],
        [bbox[2], bbox[1]],
        [bbox[0], bbox[1]],
    ]],
}

├── vern (collection)
│   ├── vern_norge_20260615 (item)
│   │   ├── vern_0_naturvern_omraade_lyrx
│   │   ├── vern_1
│   │   └── 
│   ├── vern_norge_20260615 (item)


In [46]:
# One item with many LYRX assets
item = pystac.Item(
    id=item_id,
    geometry=geom,
    bbox=bbox,
    datetime=generation_time,
    properties={
        "generation_time": generation_time.isoformat(),
        "asset_count_expected": len(fc_layers),
    },
)

for layer_name, layer_cfg in fc_layers:
    service_id = layer_cfg.get("service_id")
    lyrx_filename = f"{service_name}_{service_id}_{layer_name}.lyrx"
    lyrx_href = f"{lyrx_asset_base}/{lyrx_filename}"

    asset_key = f"{layer_name}_lyrx"
    item.add_asset(
        asset_key,
        pystac.Asset(
            href=lyrx_href,
            media_type="application/octet-stream",
            roles=["style", "metadata"],
            title=f"LYRX style for {layer_name}",
        ),
    )

print(f"Item id: {item.id}")
print(f"Assets on item: {len(item.assets)}")

Item id: vern_norge_20260615
Assets on item: 6


In [39]:
collection = pystac.Collection(
    id=collection_id,
    description=description,
    extent=pystac.Extent(
        spatial=pystac.SpatialExtent([bbox]),
        temporal=pystac.TemporalExtent([[generation_time, generation_time]]),
    ),
    license=license_name,
)

collection.add_item(item)

print(f"Collection id: {collection.id}")
print(f"Items in collection: {len(list(collection.get_items()))}")

Collection id: vern
Items in collection: 1


In [40]:
out_dir = Path("collections/vern")
out_dir.mkdir(parents=True, exist_ok=True)

collection.normalize_hrefs(str(out_dir))
collection.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)

print(f"Saved: {out_dir / 'collection.json'}")
print(f"Saved item folder: {out_dir / item.id}")

Saved: collections\vern\collection.json
Saved item folder: collections\vern\vern_norge_20260615


### Upload STAC JSON source and LYRX assets to Azure

- azcopy to `collection` to blob storage
- azcopy lyrx assest to `collections/vern/assets/lyrx`
